# When more calibration stops helping

Question: which part of uncertainty can be reduced by more draws, and which requires information about target coverage? This notebook compares frozen synthetic designs and adds two explicitly descriptive analyses. It does not pool incompatible phases or estimate a field failure rate.

Set `GENEVRA_REPLAY` to a completed replay directory. Run from a clean kernel after the archive verifier. See the research guide for exact commands.

In [ ]:
import csv
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt

run = Path(os.environ["GENEVRA_REPLAY"]).resolve()
receipt = json.loads((run / "receipt.json").read_text())
assert {"11", "12", "13", "16", "18", "19-extension"} <= set(receipt["elapsed_s"])


def csv_rows(phase, name):
    with (run / f"phase-{phase}" / name).open(newline="") as stream:
        return list(csv.DictReader(stream))


def one(rows, **fields):
    selected = [r for r in rows if all(r[k] == str(v) for k, v in fields.items())]
    if len(selected) != 1:
        raise ValueError(f"Expected one row for {fields}, got {len(selected)}")
    return selected[0]


print("Verified replay archive:", receipt["archive_sha256"])

## Sampling uncertainty under stated laws

Read phase-local median interval widths at a common look **within** each phase. The conditional tail designs use privileged sampling and, in Phase 12, a specified tail mass. Phase 13 pays for independently acquired mass and residual information. These are different information regimes, so the panels deliberately avoid a pooled effect or cost ranking.

In [ ]:
eleven = csv_rows("11", "benchmark.csv")
twelve = csv_rows("12", "benchmark.csv")
thirteen = csv_rows("13", "benchmark.csv")
panels = [
    (
        "11 · rare at .60, look 512",
        eleven,
        dict(scenario="rare", look=512, truth=0.6),
        [("bounded", {}), ("running", {})],
    ),
    (
        "12 · quiet, look 2,048",
        twelve,
        dict(scenario="quiet", look=2048),
        [("iid_betting", {}), ("strat_exact", {})],
    ),
    (
        "13 · quiet, look 2,048",
        thirteen,
        dict(scenario="quiet", look=2048),
        [("iid_betting", {"cal_rows": 0}), ("cp_residual", {"cal_rows": 131072})],
    ),
]
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), layout="constrained")
for ax, (title, rows, filters, choices) in zip(axes, panels, strict=True):
    widths = [
        float(one(rows, metric="coverage", method=method, **filters, **extra)["median_width"])
        for method, extra in choices
    ]
    ax.barh([method for method, _ in choices], widths, color="#267583")
    ax.axvline(0.6, color="#a14b37", linestyle="--")
    ax.set(title=title, xlabel="Median width", xscale="log")
plt.show()

## Shared blindness and matched access

Phase 16 shows failure of the combined audit but gives direct calibration privileged target access. Phase 18 holds the acquisition package fixed. Compare the same *denominator* within each phase. False transfer certification, mean coverage, and false threshold calls refer to different events. A full-range interval is counted as coverage even though it makes no decisive call.

In [ ]:
sixteen = csv_rows("16", "benchmark.csv")
eighteen = csv_rows("18", "rates.csv")
for rows, label, key in [(sixteen, "16", "scenario"), (eighteen, "18", "case")]:
    metrics = ("coverage", "false_certification")
    for metric in metrics:
        row = one(rows, method="combined", metric=metric, **{key: "shared_blind"})
        print(label, metric, f"{row['count']}/{row['n']}")

for metric in ("coverage_decisive_correct", "false_label", "certification"):
    r = one(eighteen, case="shared_blind", method="combined", metric=metric)
    print("18", metric, f"{r['count']}/{r['n']}")

for row in csv_rows("18", "primary.csv"):
    print(
        row["comparator"],
        round(float(row["difference"]), 4),
        tuple(round(float(row[key]), 4) for key in ("low", "high")),
    )

## Sensitivity: what no reachable budget can fix

With an exact reachable rate, unrestricted hidden tail, and missing mass at most U, the identification width is (t−c)U. The dashed lines show the necessary mass-cap threshold for width .60 under the extension's frozen span. This is algebraic sensitivity, not a new simulation. Compare the bound with actual budget-specific outcomes and the hidden-support ablation.

In [ ]:
extension = json.loads((run / "phase-19-extension/results.json").read_text())
span = 234.5 - 0.3
max_cap = 0.6 / span
caps = [i / 100_000 for i in range(501)]
fig, ax = plt.subplots(figsize=(6, 3.4), layout="constrained")
ax.plot([100 * u for u in caps], [span * u for u in caps], color="#267583")
ax.axhline(0.6, color="#a14b37", linestyle="--")
ax.axvline(100 * max_cap, color="#a14b37", linestyle="--")
ax.set(
    xlabel="Assumed upper inaccessible mass (%)",
    ylabel="Irreducible score width",
    title="Exact reachable rates, unrestricted hidden tail",
)
if figure_dir := os.environ.get("GENEVRA_NOTEBOOK_FIGURES"):
    fig.savefig(Path(figure_dir) / "identification_floor.png", dpi=180)
plt.show()
print(f"Necessary upper mass cap for width .60: {100 * max_cap:.3f}%")

for row in extension["summary"]:
    if (
        row["case"] in ("small_omission", "bad_hidden_support", "verified_hidden_support")
        and row["budget"] == 32768
        and row["method"] in ("equal_envelope", "planned_envelope", "full_hidden_support")
    ):
        measures = row["rates"]
        print(
            row["case"],
            row["method"],
            round(row["median_width"], 3),
            measures["coverage_with_precision"]["count"],
            measures["false_certificate"]["count"],
        )

## Interpretation

A narrower bound can be an advantage only under valid access and support assumptions. The invalid-support stress case makes this visible. The planner minimizes a worst-case proxy, while the benchmark scores realized precision, so the negative allocation result is informative without proving that equal spend is generally optimal. A real comparison would need defensible external bounds, a held-out target, and measured access costs.